[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter7/tool-calling.ipynb)

# Chapter 7: Tool Calling with LLMs

**Tool calling** (also known as function calling) allows LLMs to interact with external systems by generating structured function calls. Instead of just producing text, the model outputs a JSON object specifying which function to call and with what arguments.

This is foundational for building **agents** - LLMs that can take actions in the world.

## How It Works

1. You define available tools with their names, descriptions, and parameter schemas
2. The user asks a question (e.g., "What's the weather in Oakland?")
3. The LLM decides which tool to call and extracts the parameters
4. Your code executes the tool and returns results to the LLM
5. The LLM generates a final response

## Requirements

```bash
pip install openai
export OPENAI_API_KEY="your-key-here"
```

In [1]:
from openai import OpenAI

In [2]:
weather_tool = {
    "type": "function",
    "name": "get_weather",
    "description": "Get current temperature for provided coordinates in Celsius.",
    "parameters": {
        "type": "object",
        "properties": {
            "latitude": { "type": "number" },
            "longitude": { "type": "number" }
        },
        "required": ["latitude", "longitude"],
        "additionalProperties": False
    },
}

#def get_weather(latitude: float, longitude: float):
    # get the weather somehow

## Define a Tool

Tools are defined as JSON Schema objects describing the function name, description, and parameters. The LLM uses this schema to understand what the tool does and what arguments it needs.

In [3]:
client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "What's the weather like in Oakland today?"}
    ],
    functions=[weather_tool],
    function_call={"name": "get_weather"}
)

## Call the LLM with Tool

When we send a message along with available tools, the LLM will decide whether to call a tool and extract the necessary parameters from the user's query.

In [4]:
response.choices[0].message.function_call

FunctionCall(arguments='{"latitude":37.8049,"longitude":-122.2711}', name='get_weather')

> **Note:** The model returns a `function_call` object containing the function `name` (`get_weather`) and `arguments` as JSON with extracted parameters (latitude/longitude for Oakland).